# Phase 6: ML Model Training

Train Logistic Regression, Decision Tree, and KNN from the finalized five-feature table using one stratified split. This notebook does not calculate evaluation metrics or compare model performance.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "python" / "model_training.py").is_file():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "python"))

from model_training import FEATURE_COLUMNS, TARGET_COLUMN, MODEL_DIR, load_training_data, run_training, ARTIFACTS
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

## Train using the shared split

The training function validates the exact schema and binary numeric inputs, then creates one stratified 80/20 split shared by all three models. Scalers are pipeline steps and are fit only on training rows.

In [ ]:
data = load_training_data()
result = run_training(data)
print("Rows:", len(data))
print("Features:", list(FEATURE_COLUMNS))
print("Target:", TARGET_COLUMN)
print("Target counts:", data[TARGET_COLUMN].value_counts().sort_index().to_dict())
print("Train/test rows:", len(result["X_train"]), len(result["X_test"]))
print("Train/test user overlap:", len(result["train_ids"] & result["test_ids"]))

## Verify saved artifacts

Load each saved model and confirm preprocessing configuration. These are training artifact checks only; no predictions or evaluation metrics are computed.

In [ ]:
for model_name, filename in ARTIFACTS.items():
    model = joblib.load(MODEL_DIR / filename)
    if model_name == "Decision Tree":
        assert isinstance(model, DecisionTreeClassifier)
    else:
        assert isinstance(model, Pipeline)
        assert isinstance(model.named_steps["scaler"], StandardScaler)
        assert hasattr(model.named_steps["scaler"], "mean_")
    print(model_name, "loaded:", filename)

assert result["metadata"]["test_user_ids_sha256"].nunique() == 1
print("All artifacts loaded; no evaluation performed.")